# U02.E03 · Resumen ejecutable — Corpus público y clasificador de texto

INF-8239 Ciencia de Datos II · Unidad 02

Este notebook **no duplica la lógica de entrenamiento** (esa vive en `scripts/` y `src/inf8239_u02/`,
que son el código reutilizable y probado con `pytest`). Aquí solo se **reutiliza** ese código y los
artefactos ya generados para mostrar evidencia visual legible: auditoría del corpus, métricas,
matriz de confusión y análisis de errores.

Para ejecutarlo desde cero:
```
uv sync
uv run python scripts/train_text.py   # si no existen ya los artefactos en reports/ y models/
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/resumen_resultados.ipynb
```

## 1. Auditoría del corpus (datos reales)

In [ ]:
from inf8239_u02.config import ROOT, settings
from inf8239_u02.data import load_dataset, validate_dataframe, sha256

df = load_dataset()
validate_dataframe(df, settings.text_column, settings.target_column)

print("Ruta:", settings.dataset_path)
print("SHA-256:", sha256(settings.dataset_path))
print("Filas y columnas:", df.shape)
print()
print("Nulos:")
print(df[[settings.text_column, settings.target_column]].isna().sum())
print()
print("Duplicados de texto:", df[settings.text_column].astype(str).duplicated().sum())
print()
print("Distribucion de clases:")
print(df[settings.target_column].value_counts(normalize=True))

## 2. Métricas de los tres pipelines

Generadas por `scripts/train_text.py` (dummy, Naive Bayes, regresión logística).

In [ ]:
import pandas as pd

metrics = pd.read_csv(ROOT / "reports/text_metrics.csv")
metrics

## 3. Matriz de confusión (modelo seleccionado: regresión logística)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(ROOT / "reports/confusion_text.png")))

## 4. Análisis de errores (74 casos categorizados)

Categorías asignadas tras leer el texto completo de cada error — ver `docs/analisis_errores.md` para la interpretación completa.

In [ ]:
errors = pd.read_csv(ROOT / "reports/error_analysis.csv")
print("Total de errores en el conjunto de prueba:", len(errors))
errors["category"].value_counts()

In [ ]:
pd.set_option("display.max_colwidth", 160)
errors.groupby("category").apply(lambda g: g.head(2))[["real", "predicted", "text"]]

## 5. Modelo guardado: predicciones de verificación

Incluye los tres casos fuera de lo esperado que se probaron manualmente en la app de Streamlit (español, inglés casual fuera de dominio, inglés formal corto) para mostrar que el modelo nunca se abstiene — siempre fuerza una respuesta entre las dos clases.

In [ ]:
import joblib

model = joblib.load(ROOT / "models/text_model.joblib")

casos = [
    "Excelente servicio",
    "Today, we are waiting for the news from London, while we are waiting, we must go for a pizza.",
    "I believe that students should not be allowed to use cell phones during class because it distracts them from learning and reduces their attention span.",
]
for texto, pred in zip(casos, model.predict(casos)):
    print(f"[{pred}] {texto}")

## 6. Síntesis

Ver la sección **"Cierre interpretativo"** en `README.md` para la interpretación completa (resultado principal, clase con mayor dificultad, limitaciones del dataset y decisión antes del despliegue).